In [1]:
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
import pandas as pd

In [29]:
def class_distance_separation(X, y, metric="cosine", sample_pairs=50000, seed=42):
    rng = np.random.default_rng(seed)

    X = np.asarray(X)
    y = np.asarray(y)

    if metric == "cosine":
        X = normalize(X)

    n = len(X)

    i = rng.integers(0, n, size=sample_pairs)
    j = rng.integers(0, n, size=sample_pairs)

    mask = i != j
    i, j = i[mask], j[mask]

    dists = np.linalg.norm(X[i] - X[j], axis=1) if metric == "euclidean" else (
        1 - np.sum(X[i] * X[j], axis=1)
    )

    same = y[i] == y[j]

    within = dists[same]
    between = dists[~same]

    return pd.DataFrame([{
        "metric": metric,
        "within_mean": within.mean(),
        "within_std": within.std(),
        "between_mean": between.mean(),
        "between_std": between.std(),
        "separation_gap": between.mean() - within.mean(),
        "relative_gap": (between.mean() - within.mean()) / between.mean(),
    }])

In [26]:
import os
import sys
module_path = "../osr_tiny/outputs_osr/"
if module_path not in sys.path:
    sys.path.append(module_path)

OUT_DIR = "../osr_tiny/outputs_osr"

train_machine_feat_path_tiny = os.path.join(OUT_DIR, 'vgg32_train_reps.csv')
train_machine_feat_path_cifar = r'cifar10_train_feat_vit.csv'
train_machine_feat_path_mnist = r'mnist_train_id_0_5_feat_cnn.csv'
train_machine_feat_path_svhn = r'svhn_train_id_feat_wrn.csv'
train_machine_feat_path_mp10 =r'dgl_train_feat.csv'

In [27]:
df_mp10 = pd.read_csv(train_machine_feat_path_mp10)

In [5]:
df_cifar = pd.read_csv(train_machine_feat_path_cifar)

In [10]:
df_mnist = pd.read_csv(train_machine_feat_path_mnist)

In [16]:
df_svhn = pd.read_csv(train_machine_feat_path_svhn)

In [20]:
df_tiny = pd.read_csv(train_machine_feat_path_tiny)

In [30]:
# first column = label
label_col = df_cifar.columns[0]

y = df_cifar[label_col].astype(int).to_numpy()
X = df_cifar.drop(columns=[label_col]).astype(float).to_numpy()

result_cifar = class_distance_separation(X, y, metric="cosine")
result_cifar

,metric,within_mean,within_std,between_mean,between_std,separation_gap,relative_gap
0,cosine,0.08888,0.079931,1.289022,0.088528,1.200143,0.931049


In [31]:
# first column = label
label_col = df_mnist.columns[0]

y = df_mnist[label_col].astype(int).to_numpy()
X = df_mnist.drop(columns=[label_col]).astype(float).to_numpy()

result_mnist = class_distance_separation(X, y, metric="cosine")
result_mnist

,metric,within_mean,within_std,between_mean,between_std,separation_gap,relative_gap
0,cosine,0.121563,0.083086,0.616945,0.135488,0.495382,0.802959


In [32]:
label_col = df_svhn.columns[0]

y = df_svhn[label_col].astype(int).to_numpy()
X = df_svhn.drop(columns=[label_col]).astype(float).to_numpy()

result_svhn = class_distance_separation(X, y, metric="cosine")
result_svhn

,metric,within_mean,within_std,between_mean,between_std,separation_gap,relative_gap
0,cosine,0.05635,0.045049,0.594543,0.107355,0.538193,0.905221


In [33]:
y = df_tiny["label"].astype(int).to_numpy()

X = df_tiny.drop(columns=["label"]).astype(float).to_numpy()

result_tiny = class_distance_separation(X, y, metric="cosine")

result_tiny

,metric,within_mean,within_std,between_mean,between_std,separation_gap,relative_gap
0,cosine,0.071582,0.043796,0.65189,0.113566,0.580308,0.890193


In [35]:
label_col = df_mp10.columns[0]

y = df_mp10[label_col].astype(int).to_numpy()
X = df_mp10.drop(columns=[label_col]).astype(float).to_numpy()

result_mp10 = class_distance_separation(X, y, metric="cosine")
result_mp10

,metric,within_mean,within_std,between_mean,between_std,separation_gap,relative_gap
0,cosine,0.07849,0.154912,0.520316,0.258107,0.441826,0.84915


In [36]:
result_cifar["dataset"] = "CIFAR10"
result_mnist["dataset"] = "MNIST"
result_tiny["dataset"] = "TinyImageNet"
result_mp10["dataset"] = "MP10"
result_svhn["dataset"] = "SVHN"

all_results = pd.concat([
    result_cifar,
    result_mnist,
    result_tiny,
    result_mp10,
    result_svhn
], ignore_index=True)

all_results

,metric,within_mean,within_std,between_mean,between_std,separation_gap,relative_gap,dataset
0,cosine,0.088880,0.079931,1.289022,0.088528,1.200143,0.931049,CIFAR10
1,cosine,0.121563,0.083086,0.616945,0.135488,0.495382,0.802959,MNIST
2,cosine,0.071582,0.043796,0.651890,0.113566,0.580308,0.890193,TinyImageNet
3,cosine,0.078490,0.154912,0.520316,0.258107,0.441826,0.849150,MP10
4,cosine,0.056350,0.045049,0.594543,0.107355,0.538193,0.905221,SVHN
